# Data Preparation

Data Preparation for Analyses

Tristan Muno [](https://orcid.org/0009-0002-3078-8436) (University of Mannheim)  
Vera Okisheva (University of Mannheim)  
Raphael Klöckner (University of Mannheim)  
June 12, 2026

Goal: create long format analysis tibble with outcome, predictors, respondent and country IDs.

# Setup

In [ ]:
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr etc
  "here", # to never worry about directories
  "ggpubr", # theme_pubr()
  "janitor", # tabyl()
  "knitr", # for kable()
  "sessioninfo" # session docs
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)


Loading required package: tidyverse

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: here

here() starts at C:/R/research/MBEU25

Loading required package: ggpubr

Loading required package: janitor


Attaching package: 'janitor'


The following objects are masked from 'package:stats':

    chisq.test, fisher.test


Loading required package: knitr

Loading required package: sessioninfo

  tidyverse        here      ggpubr     janitor       knitr sessioninfo 
       TRUE        TRUE        TRUE        TRUE        TRUE        TRUE 

# Load Data

Full dataset (and pre-processing pipeline) is available at [GitHub](https://github.com/LS-Konig/eu25games2019). We can directly fetch the data from there (by now also compressed, so only ca 30 MB). `read_rds()` can decompress the file directly.

In [ ]:
# set to true do fetch from source repo
download_fresh <- F

if (download_fresh) {
  # URL of raw data
  raw_url <- "https://raw.githubusercontent.com/LS-Konig/eu25games2019/main/data/03_final/eu25games2019.rds?cache_bust=1"

  # Create temporary file path
  tmp <- tempfile(fileext = ".rds")

  # Download binary file locally
  download.file(raw_url, tmp, mode = "wb")

  # Read data into R (takes a couple seconds as it decompresses the file)
  eu25games2019 <- read_rds(tmp)

  # Write (compressed) file to data/01_raw
  write_rds(
    eu25games2019,
    file = here(
      "data",
      "01_raw",
      "eu25games2019.rds"
    ),
    compress = "xz"
  )

  # Clean up variables
  rm(raw_url, tmp)

  write
} else {
  eu25games2019 <- read_rds(
    file = here(
      "data",
      "01_raw",
      "eu25games2019.rds"
    )
  )
}


The conjoint partner nationality is now split into the value *shown* on screen (`cj_<slot>_nationality_shown`) and the background lottery that was *drawn* (`cj_<slot>_nationality_drawn`, plus its raw string `..._drawn_raw`). These replace the old conflated `cj_<slot>_nat` / `cj_<slot>_nat_raw`. The check below confirms the rename landed across all six profile slots and that no game columns leak into Wave 1.

In [ ]:
slots <- c(
  "cj_dict1",
  "cj_dict2",
  "cj_dict3",
  "cj_trust1",
  "cj_trust2",
  "cj_trust3"
)
new_cols <- as.vector(outer(
  slots,
  c("_nationality_drawn", "_nationality_drawn_raw", "_nationality_shown"),
  paste0
))
old_cols <- c(
  paste0(slots, "_nat"),
  paste0(slots, "_nat_raw"),
  paste0("der_", slots, "_nationality")
)

stopifnot(
  # dimensions roughly stable (rename is net-neutral on column count)
  abs(nrow(eu25games2019) - 103685) < 50,
  abs(ncol(eu25games2019) - 847) < 5,
  # 18 new shown/drawn columns present
  all(new_cols %in% names(eu25games2019)),
  # no old conflated names survive
  !any(old_cols %in% names(eu25games2019)),
  # no dictator/trust games in wave 1: all game-slot columns NA there
  # (the separate cj_task* / cj_rand* conjoint instrument does run in wave 1)
  all(vapply(
    eu25games2019[
      eu25games2019$meta_wave == 1,
      grepl("^cj_(dict|trust)[123]_", names(eu25games2019))
    ],
    \(x) all(is.na(x)),
    logical(1)
  ))
)


# Filter out wave 1 only respondets

Data comes in three waves with partial panel structure.

In [ ]:
# count obs per wave
cat("Raw observations per wave:\n")


Raw observations per wave:

# A tibble: 3 × 2
  meta_wave     n
      <dbl> <int>
1         1 44857
2         2 25767
3         3 33061


Raw observations total:

[1] 103685


Missing values in ID variables:

...

# A tibble: 1 × 2
  duplicated_pids excess_rows
            <int>       <dbl>
1             975        1153


Panelists by number of distinct waves participated:

# A tibble: 3 × 2
  n_waves n_panelists
    <int>       <int>
1       1       59915
2       2       11818
3       3        6327


Panelists appearing in 2+ waves:

[1] 18145

# Filter our respondents who did not participate in games (w2 and w3 only)

In [ ]:
# pids of anyone who participated in at least one of w2 / w3
pids_in_games <- eu25games2019 |>
  filter(meta_wave %in% c(2, 3)) |>
  distinct(meta_pid) |>
  pull(meta_pid)

# keep all w2 + w3 rows; from w1 keep only pids that reappear in w2 or w3
eu25games2019 <- eu25games2019 |>
  # n = 103,685 -> 70,685 (dropped 33,000: wave-1-only respondents who never played a game)
  filter(meta_wave %in% c(2, 3) | meta_pid %in% pids_in_games)

cat("Observations remaining:\n")


Observations remaining:

[1] 70685


Observations per wave:

# A tibble: 3 × 2
  meta_wave     n
      <dbl> <int>
1         1 11857
2         2 25767
3         3 33061


Distinct panelists remaining:

[1] 45321

# Select clean covariate candidates

In [ ]:
eu25games2019 |>
  select(-contains("_raw")) |>
  select(contains("q_")) |>
  glimpse()


Rows: 70,685
Columns: 305
$ q_gender                                   <chr> "male", "male", "male", "fe…
$ q_age                                      <int> 26, 16, 64, 37, 65, 58, 62,…
$ q_nationality                              <chr> "Austria", "Belgium", "Aust…
$ q_nationality_other                        <chr> NA, NA, NA, NA, NA, NA, NA,…
$ q_identity_country                         <int> 1, 5, 1, 1, 1, 2, 1, 1, 1, …
$ q_identity_eu                              <int> 2, 5, 2, 2, 3, 4, 2, 1, 1, …
$ q_identity_europe                          <int> 2, 5, 2, 2, 2, 3, 1, 1, 1, …
$ q_pop_will                                 <int> 1, NA, NA, 6, 7, NA, 7, 7, …
$ q_pop_people                               <int> NA, NA, 4, 5, NA, NA, 1, 7,…
$ q_pop_citizen                              <int> NA, NA, 6, NA, NA, 4, 2, 7,…
$ q_pop_reps                                 <int> NA, NA, 6, NA, 6, NA, 7, 7,…
$ q_pop_goodevil                             <int> NA, NA, NA, NA, 5, 4, NA, N…
$ q_pop_compro

**NOTE:** We have decided to only include items asked *prior* to the games.

We can keep all questions asked in wave 1, as those games only started in wave 2.

In [ ]:
eu25games2019 |>
  filter(meta_wave == 1) |>
  select(-contains("_raw")) |>
  select(contains("q_")) |>
  glimpse()


Rows: 11,857
Columns: 305
$ q_gender                                   <chr> "male", "male", "male", "fe…
$ q_age                                      <int> 26, 16, 64, 37, 65, 58, 62,…
$ q_nationality                              <chr> "Austria", "Belgium", "Aust…
$ q_nationality_other                        <chr> NA, NA, NA, NA, NA, NA, NA,…
$ q_identity_country                         <int> 1, 5, 1, 1, 1, 2, 1, 1, 1, …
$ q_identity_eu                              <int> 2, 5, 2, 2, 3, 4, 2, 1, 1, …
$ q_identity_europe                          <int> 2, 5, 2, 2, 2, 3, 1, 1, 1, …
$ q_pop_will                                 <int> 1, NA, NA, 6, 7, NA, 7, 7, …
$ q_pop_people                               <int> NA, NA, 4, 5, NA, NA, 1, 7,…
$ q_pop_citizen                              <int> NA, NA, 6, NA, NA, 4, 2, 7,…
$ q_pop_reps                                 <int> NA, NA, 6, NA, 6, NA, 7, 7,…
$ q_pop_goodevil                             <int> NA, NA, NA, NA, 5, 4, NA, N…
$ q_pop_compro

These can serve as fillup maybe.

Variables in w2 and w3 before games (Source: [Data GitHub Repo](https://github.com/LS-Konig/eu25games2019)):

In [ ]:
vars_pre_game <- c(
  "q_gender",
  "q_age",
  "q_nationality",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_euint_pos",
  "q_eu_efficacy_understand",
  "q_eu_efficacy_care",
  "q_eu_efficacy_nosay",
  "q_pop_will",
  "q_pop_people",
  "q_pop_citizen",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_election_challenge_polgroups",
  "q_election_impact_eu",
  "q_election_importance_self"
)

# ignoring vote choice and party IDs for now


| Variable | Coverage | Question wording |
|:---------------------|:--------------------------:|:---------------------|
| `q_gender` | 2/3 | Please indicate your gender. |
| `q_age` | 2/3 | How old are you? |
| `q_nationality` | 2/3 | What is your nationality? Please tell me the country(ies) that applies(y). |
| `q_identity_country` | 2/3 | Please tell me how attached you feel to… - United Kingdom |
| `q_identity_eu` | 2/3 | Please tell me how attached you feel to… - The European Union (EU) |
| `q_identity_europe` | 2/3 | Please tell me how attached you feel to… - Europe |
| `q_religion` | 2/3 | Do you consider yourself to be…? |
| `q_class` | 2/3 | Do you see yourself and your household belonging to…? |
| `q_euint_pos` | 2/3 | Some people say that European integration should be pushed further. Others say it already has gone too far. What is your opinion? Please indicate your views using a scale from 1 to 7, where ‘1’ means European integration “has already gone too far” and ‘7’ means it “should be pushed further”. What number on this scale best describes your position? |
| `q_eu_efficacy_understand` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - I can understand and evaluate important political questions about the EU |
| `q_eu_efficacy_care` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - The politicians in the EU care about what people like me think |
| `q_eu_efficacy_nosay` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - People like me don’t have any say about what the EU does |
| `q_pop_will` | 2/3 | Please indicate how much you agree with each of the following statements. - The politicians should follow the will of the people. |
| `q_pop_people` | 2/3 | Please indicate how much you agree with each of the following statements. - The people, and not politicians, should make our most important policy decisions. |
| `q_pop_citizen` | 2/3 | Please indicate how much you agree with each of the following statements. - I would rather be represented by a citizen than by a specialized politician. |
| `q_pop_reps` | 2/3 | Please indicate how much you agree with each of the following statements. - Elected officials talk too much and take too little action. |
| `q_pop_goodevil` | 2/3 | Please indicate how much you agree with each of the following statements. - Politics is ultimately a struggle between good and evil. |
| `q_pop_compromise` | 2/3 | Please indicate how much you agree with each of the following statements. - What people call “compromise” in politics is really just selling out on one’s principles. |
| `q_dem_compromise` | 2/3 | Please indicate how much you agree with each of the following statements. - In a democracy it is important to make compromises among differing viewpoints. |
| `q_dem_listen` | 2/3 | Please indicate how much you agree with each of the following statements. - It is important to listen to the opinion of other groups. |
| `q_tech_experts` | 2/3 | Please indicate how much you agree with each of the following statements. - Our country would run better if political decisions were left up to experts instead of politicians and citizens. |
| `q_tech_leaders` | 2/3 | Please indicate how much you agree with each of the following statements. - The leaders of our country should be more educated and skilled than ordinary citizens. |
| `q_party_harm` | 2/3 | Please indicate how much you agree with each of the following statements. - Political parties do more harm than good to society |
| `q_people_incompetent` | 2/3 | Please indicate how much you agree with each of the following statements. - Ordinary people don’t know what policies are good for them. |
| `q_eu_longterm` | 2/3 | In thinking about the European Union (EU) and its decisions more generally, some people argue that it should act upon its long-term goals without pandering to the demand of the people. On the other hand, others argue that the EU should listen to the demand of people, rather than to act upon its long-term goals. When it comes to the question of whether the EU should act upon its long-term goals or it should listen to the demand of the people, what is your opinion? - The EU should act upon its long-term goals |
| `q_eu_responsive` | 2/3 | In thinking about the European Union (EU) and its decisions more generally, some people argue that it should act upon its long-term goals without pandering to the demand of the people. On the other hand, others argue that the EU should listen to the demand of people, rather than to act upon its long-term goals. When it comes to the question of whether the EU should act upon its long-term goals or it should listen to the demand of the people, what is your opinion? - The EU should listen to the demand of the people |
| `q_eu_imp_nat_econ` | 2/3 | Could you tell me whether you think that the EU has a rather positive or rather negative effect on each of the following domains? - UK economy |
| `q_eu_imp_nat_cul` | 2/3 | Could you tell me whether you think that the EU has a rather positive or rather negative effect on each of the following domains? - Our culture and identity |
| `q_eu_imp_nat_pol` | 2/3 | Could you tell me whether you think that the EU has a rather positive or rather negative effect on each of the following domains? - UK’s political status and influence |
| `q_eu_abolish` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - If the EU started making a lot of decisions that most people disagreed with, it might be better to do away with the EU altogether |
| `q_eu_satisfaction` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - I am satisfied with the performance of the EU |
| `q_election_challenge_polgroups` | 2 | What is your opinion? - How likely do you think the upcoming elections will challenge the existing political groups and members of the European Parliament? |
| `q_election_impact_eu` | 2 | What is your opinion? - How likely do you think the upcoming elections will play a big role in shaping the European Parliament and the EU’s next five years? |
| `q_election_importance_self` | 2 | How important is the outcome of the upcoming election to you personally? |

# Clean incomplete responses, failed attention checks

In [ ]:
eu25games2019_clean <- eu25games2019 |>
  # only respondents who passed attention check
  # n = 70,685 -> 41,457 (dropped 29,228: failed attention check der_att_check_3)
  filter(der_att_check_3)

# still pid duplicates?
eu25games2019_clean |>
  group_by(meta_wave) |>
  count(meta_pid) |>
  count(n)


Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.

# A tibble: 10 × 3
# Groups:   meta_wave [3]
   meta_wave     n    nn
       <dbl> <int> <int>
 1         1     1  7617
 2         1     2    19
 3         1    21     1
 4         2     1 15160
 5         2     2    69
 6         2     7     1
 7         3     1 18122
 8         3     2   163
 9         3     3     6
10         3    10     1

Storing counts in `nn`, as `n` already present in input
ℹ Use `name = "new_name"` to pick a new name.

# A tibble: 6 × 3
# Groups:   meta_wave [2]
  meta_wave     n    nn
      <dbl> <int> <int>
1         2     1 13960
2         2     2    16
3         2     6     1
4         3     1 16510
5         3     2    32
6         3     8     1

# A tibble: 50 × 3
# Groups:   meta_wave [2]
   meta_wave meta_pid       n
       <dbl> <chr>      <int>
 1         3 1575599903     8
 2         2 <NA>           6
 3         2 1067766259     2
 4         2 1147353522     2
 5         2 1275872150     2
 6         2 1383958152     2
 7         2 1385201604     2
 8         2 1416296382     2
 9         2 1463470715     2
10         2 1476773133     2
# ℹ 40 more rows

# A tibble: 9 × 3
# Groups:   meta_wave, meta_pid [9]
  meta_wave meta_pid       n
      <dbl> <chr>      <int>
1         3 1575599903     6
2         2 1275872150     2
3         2 1385201604     2
4         2 1416296382     2
5         3 1128065675     2
6         3 1415945402     2
7         3 1451625374     2
8         3 1470826975     2
9         3 1472281906     2

# A tibble: 2 × 2
  meta_responseid   total_na
  <chr>                <int>
1 R_1HhimczSQNnABIG      422
2 R_2q1R9Wp33jWr7hv      424

# A tibble: 2 × 2
  meta_responseid   total_na
  <chr>                <int>
1 R_0CERPAMUI0Og56V      434
2 R_2t4q1GTfVbXoWPc      440

# A tibble: 2 × 2
  meta_responseid   total_na
  <chr>                <int>
1 R_3frzoiOPtnaah6x      435
2 R_Wv4d0yluMW6fAFX      432

# Inspect missings in selected covariates

Because the games only start in wave 2, every pre-game item asked in wave 1 is a valid pre-treatment measure for the same panelist. We therefore carry each panelist’s wave-1 value forward into their *missing* wave-2/3 cells, matched on `(meta_pid, meta_country)`. Only `NA` cells are touched — existing values are never overwritten — and every filled cell is flagged so the operation is fully auditable. The recovery is almost entirely attitudinal items (EU-efficacy and populism scales): carrying them forward assumes temporal stability, which is acceptable here because the waves are close together and these enter the analysis only as respondent-level moderators.

In [ ]:
# w1 items available as fill source
eu25games2019_clean2 |>
  filter(meta_wave == 1) |>
  select(all_of(vars_pre_game)) |>
  glimpse()


Rows: 5,911
Columns: 34
$ q_gender                       <chr> "female", "male", "male", "female", "ma…
$ q_age                          <int> 37, 65, 58, 35, 43, 56, 56, 20, 56, 60,…
$ q_nationality                  <chr> "Austria", "Austria", "Austria", "Austr…
$ q_identity_country             <int> 1, 1, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 2, …
$ q_identity_eu                  <int> 2, 3, 4, 2, 3, 5, 2, 3, 3, 1, 3, 5, 2, …
$ q_identity_europe              <int> 2, 2, 3, 3, 3, 3, 1, 2, 2, 1, 2, 4, 2, …
$ q_religion                     <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ q_class                        <chr> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ q_euint_pos                    <int> NA, NA, NA, NA, NA, NA, NA, NA, NA, NA,…
$ q_eu_efficacy_understand       <int> 4, 1, 2, 4, 5, 4, 4, 6, 5, 6, 4, 6, 5, …
$ q_eu_efficacy_care             <int> 3, NA, NA, 3, 2, NA, 3, 2, 4, NA, 1, 1,…
$ q_eu_efficacy_nosay            <int> NA, 2, 5, NA, NA, 5, NA, NA, NA, 3, NA,…
$ q_pop_will    

  variable                na_before   na_after   n_filled
  --------------------- ----------- ---------- ----------
  q_eu_efficacy_care          14403      12099       2304
  q_eu_efficacy_nosay         14279      12134       2145
  q_pop_will                   9340       7646       1694
  q_pop_people                 9360       7703       1657
  q_pop_citizen                9338       7713       1625
  q_age                         275        204         71
  q_pop_reps                    150        127         23

  : Wave-1 backfill: W2/W3 missing cells before/after and cells
  recovered.


Total cells filled from wave 1: 9519 

   meta_wave     meta_country         meta_pid           q_gender        
 Min.   :1.000   Length:35212       Length:35212       Length:35212      
 1st Qu.:2.000   Class :character   Class :character   Class :character  
 Median :2.000   Mode  :character   Mode  :character   Mode  :character  
 Mean   :2.283                                                           
 3rd Qu.:3.000                                                           
 Max.   :3.000                                                           
                                                                         
     q_age        q_nationality      q_identity_country q_identity_eu  
 Min.   : 18.00   Length:35212       Min.   :1.000      Min.   :1.000  
 1st Qu.: 35.00   Class :character   1st Qu.:1.000      1st Qu.:2.000  
 Median : 47.00   Mode  :character   Median :1.000      Median :3.000  
 Mean   : 46.31                      Mean   :1.571      Mean   :2.771  
 3rd Qu.: 58.00                      3rd Qu.:2.0

# Reshape to long format data for analyses

`eu25games2019` contains data in wide format, i.e., one row corresponds one survey response. To analyze outcomes, the data need to be transformed to long (tidy) format, with one row equal to one observation, in this case one row should be one game played.

In [ ]:
# select relevant variables
eu25_long <- eu25games2019_clean_selected_final |>
  # to long format
  # n = 28,201 -> 169,206 (reshape, not a drop: x6 = 2 games x 3 rounds; 84,603 rows per game)
  pivot_longer(
    cols = -c(
      starts_with("meta_"),
      starts_with("q_"),
      starts_with("cj_attr"),
      starts_with("cj_treatment"),
      contains("_instr_")
    ),
    names_to = c("game_type", "round", ".value"),
    names_pattern = "^(cj_dict|cj_trust)(\\d+)_(.+)$"
  ) |>
  # keep cj_ prefix for experimental vars
  rename_with(
    ~ paste0("cj_", .),
    -c(
      starts_with("meta_"),
      starts_with("q_"),
      starts_with("cj_attr"),
      starts_with("cj_treatment"),
      contains("_instr_")
    )
  )


# Display-aware profile covariates

The four treatment conditions in `cj_treatment` govern *which* profile features were actually shown to the respondent. EU stance was displayed only in conditions whose name contains `eustance` (treatments 1 and 3); partisan affiliation was displayed only in conditions whose name contains `partisan` (treatments 1 and 2). The raw conjoint columns `cj_eupos` and `cj_party_raw` carry a behind-the-scenes value in *every* condition, including those where the feature was never displayed, so the displayed contrast has to be reconstructed from `cj_treatment`. The partner nationality is the displayed `cj_nationality_shown` (not the background `cj_nationality_drawn` lottery, which is overridden by the treatment branch in every condition except `4_somenat`).

We therefore derive two display-aware profile covariates with the `der_cj_` prefix. `der_cj_eu_identity` collapses to three levels: the shown EU-citizen contrast where EU stance was displayed, and a single `not_displayed` level otherwise. `der_cj_partisanship` contrasts conational profiles where party was shown (`shown`, treatments 1 and 2) against the apples-to-apples conational profile where party was suppressed (`not_shown`, treatment 4 with the displayed nationality equal to the respondent’s country, i.e. `cj_nationality_shown == "own_country"`); every other profile is `not_applicable`.

In [ ]:
eu25_long <- eu25_long |>
  mutate(
    # eu stance shown -> 1,3
    der_cj_eu_identity = case_when(
      grepl("eustance", cj_treatment) & cj_eupos == "eu_citizen" ~ "eu_citizen",
      grepl("eustance", cj_treatment) & cj_eupos == "not_eu_citizen" ~
        "not_eu_citizen",
      !grepl("eustance", cj_treatment) ~ "not_displayed"
    ),
    # party shown -> 1,2; conational suppressed -> 4 & own_country
    der_cj_partisanship = case_when(
      grepl("partisan", cj_treatment) ~ "shown",
      cj_treatment == "4_somenat" &
        cj_nationality_shown == "own_country" ~ "not_shown",
      .default = "not_applicable"
    )
  )


The cross-tabs below confirm the display logic: `der_cj_eu_identity` is `not_displayed` in exactly the two no-`eustance` conditions, and `der_cj_partisanship` separates the shown-party conationals from the treatment-4 suppressed conationals.

In [ ]:
count(eu25_long, cj_treatment, der_cj_eu_identity) |>
  kable(caption = "der_cj_eu_identity by treatment condition.")
count(eu25_long, cj_treatment, cj_nationality_shown, der_cj_partisanship) |>
  kable(
    caption = "der_cj_partisanship by treatment condition and shown nationality."
  )


  cj_treatment                der_cj_eu_identity         n
  --------------------------- -------------------- -------
  1_conat_eustance_partisan   eu_citizen             30702
  1_conat_eustance_partisan   not_eu_citizen         30835
  2_conat_partisan            not_displayed          30899
  3_eunat_eustance            eu_citizen             15326
  3_eunat_eustance            not_eu_citizen         15330
  4_somenat                   not_displayed          46114


  --------------------------------------------------------------------------------
  cj_treatment                cj_nationality_shown   der_cj_partisanship         n
  --------------------------- ---------------------- --------------------- -------
  1_conat_eustance_partisan   own_country            shown                   61537

  2_conat_partisan            own_country            shown                   30899

  3_eunat_eustance            eu                     not_applicable          30656

  4_somenat                   eu                     not_applicable          11462

  4_somenat                   non_eu                 not_applicable          11648

  4_somenat                   own_country            not_shown               23004
  --------------------------------------------------------------------------------


In [ ]:
stopifnot(
  # no missingness introduced
  !anyNA(eu25_long$der_cj_eu_identity),
  !anyNA(eu25_long$der_cj_partisanship),
  # not_displayed iff eu stance not shown
  all(
    (eu25_long$der_cj_eu_identity == "not_displayed") ==
      !grepl("eustance", eu25_long$cj_treatment)
  ),
  # shown iff party displayed condition
  all(
    (eu25_long$der_cj_partisanship == "shown") ==
      grepl("partisan", eu25_long$cj_treatment)
  ),
  # not_shown is exactly the suppressed conational counterfactual
  all(
    (eu25_long$der_cj_partisanship == "not_shown") ==
      (eu25_long$cj_treatment == "4_somenat" &
        eu25_long$cj_nationality_shown == "own_country")
  )
)


The shown nationality is treatment-determined: co-national in the two conational conditions, EU in the EU-stance condition, and equal to the drawn lottery only in `4_somenat`. This is the displayed-profile contrast the moderator analyses use, in contrast to the flat drawn lottery.

In [ ]:
count(eu25_long, cj_treatment, cj_nationality_shown) |>
  kable(
    caption = "Shown nationality by treatment condition (now treatment-dependent)."
  )


  cj_treatment                cj_nationality_shown         n
  --------------------------- ---------------------- -------
  1_conat_eustance_partisan   own_country              61537
  2_conat_partisan            own_country              30899
  3_eunat_eustance            eu                       30656
  4_somenat                   eu                       11462
  4_somenat                   non_eu                   11648
  4_somenat                   own_country              23004


In [ ]:
shown <- eu25_long$cj_nationality_shown
drawn <- eu25_long$cj_nationality_drawn
trt <- eu25_long$cj_treatment

stopifnot(
  # conational conditions display own_country
  all(shown[grepl("conat", trt)] == "own_country"),
  # eu-stance-only condition displays eu
  all(shown[trt == "3_eunat_eustance"] == "eu"),
  # somenat passes the drawn lottery through unchanged
  all(shown[trt == "4_somenat"] == drawn[trt == "4_somenat"]),
  # shown is no longer the flat lottery: it varies by treatment
  dplyr::n_distinct(shown[grepl("conat", trt)]) == 1
)


# Descriptives

In [ ]:
# respondent covariates --------------------------------------------------------
# gender
skimr::skim(eu25_long)


  -----------------------------------------------------------------------------------------------------------------
  skim_variable                 n_missing   complete_rate    mean        sd   p0   p25   p50   p75     p100 hist
  --------------------------- ----------- --------------- ------- --------- ---- ----- ----- ----- -------- -------
  meta_wave                             0               1    2.54      0.50    2     2     3     3        3 ▇▁▁▁▇

  q_age                                 0               1   46.10     14.33   18    35    47    58       99 ▆▇▇▂▁

  q_identity_country                    0               1    1.56      0.89    1     1     1     2        5 ▇▃▁▁▁

  q_identity_eu                         0               1    2.76      1.19    1     2     3     4        5 ▃▇▇▃▃

  q_identity_europe                     0               1    2.31      1.03    1     2     2     3        5 ▅▇▅▂▁

  q_eu_efficacy_understand              0               1    4.67      1.36    1     4     5     6        7 ▂▂▅▇▆

  q_pop_reps                            0               1    5.51      1.30    1     5     6     7        7 ▁▁▂▃▇

  q_pop_goodevil                        0               1    4.11      1.61    1     3     4     5        7 ▅▃▇▅▆

  q_pop_compromise                      0               1    4.53      1.54    1     4     5     6        7 ▃▃▇▇▇

  q_dem_compromise                      0               1    5.38      1.10    1     5     5     6        7 ▁▁▂▆▇

  q_dem_listen                          0               1    5.89      0.98    1     5     6     7        7 ▁▁▁▂▇

  q_tech_experts                        0               1    4.73      1.52    1     4     5     6        7 ▂▂▆▆▇

  q_tech_leaders                        0               1    5.48      1.36    1     5     6     7        7 ▁▁▂▃▇

  q_party_harm                          0               1    4.61      1.58    1     4     5     6        7 ▃▃▇▆▇

  q_people_incompetent                  0               1    4.26      1.64    1     3     4     5        7 ▆▅▇▇▇

  q_eu_longterm                         0               1    4.87      1.39    1     4     5     6        7 ▂▂▅▇▇

  q_eu_responsive                       0               1    5.62      1.13    1     5     6     7        7 ▁▁▂▅▇

  q_eu_imp_nat_econ                     0               1    2.71      1.09    1     2     3     3        5 ▂▇▆▃▂

  q_eu_imp_nat_cul                      0               1    3.03      1.05    1     2     3     4        5 ▁▆▇▅▂

  q_eu_imp_nat_pol                      0               1    3.00      1.05    1     2     3     4        5 ▂▆▇▅▂

  q_eu_abolish                          0               1    4.50      1.70    1     3     5     6        7 ▃▃▆▆▇

  q_eu_satisfaction                     0               1    3.95      1.52    1     3     4     5        7 ▆▅▇▇▅

  cj_dict_instr_firstclick              0               1    7.25    552.40    0     0     0     0    92159 ▇▁▁▁▁

  cj_dict_instr_lastclick               0               1    8.68    556.47    0     0     0     0    92171 ▇▁▁▁▁

  cj_dict_instr_pagesubmit              0               1   84.58   1892.64    0    14    29    46   271352 ▇▁▁▁▁

  cj_dict_instr_clickcount              0               1    0.23      1.80    0     0     0     0      100 ▇▁▁▁▁

  cj_trust_instr_firstclick             0               1    9.30    145.89    0     0     0     0    19972 ▇▁▁▁▁

  cj_trust_instr_lastclick              0               1   15.91    171.24    0     0     0     0    19972 ▇▁▁▁▁

  cj_trust_instr_pagesubmit             0               1   87.43   2758.99    1    12    30    69   456570 ▇▁▁▁▁

  cj_trust_instr_clickcount             0               1    1.01      4.01    0     0     0     0      120 ▇▁▁▁▁

  q_edu_age_stop                        0               1   22.21      6.70    0    18    21    24       83 ▁▇▁▁▁

  cj_pl1                                0               1    6.74      2.38    0     5     7     9       10 ▁▁▇▅▆

  cj_pl2                                0               1    3.26      2.38    0     1     3     5       10 ▇▅▆▁▁

  cj_firstclick                         0               1   20.38    252.68    0     5     9    18    56351 ▇▁▁▁▁

  cj_lastclick                          0               1   29.09    265.78    0     8    15    28    56359 ▇▁▁▁▁

  cj_pagesubmit                         0               1   32.77    266.60    1    11    18    32    56360 ▇▁▁▁▁

  cj_clickcount                         0               1    2.84      2.57    0     2     2     3      146 ▇▁▁▁▁

  cj_age                                0               1   41.59     16.53   18    30    42    53       65 ▇▇▇▇▇
  -----------------------------------------------------------------------------------------------------------------


# Save long format data

In [ ]:
write_rds(
  eu25_long,
  file = here(
    "data",
    "02_processed",
    "eu25_long.rds"
  ),
  compress = "xz"
)


# Session Info

In [ ]:
session_info()


─ Session info ───────────────────────────────────────────────────────────────
 setting  value
 version  R version 4.5.3 (2026-03-11 ucrt)
 os       Windows 11 x64 (build 26200)
 system   x86_64, mingw32
 ui       RTerm
 language (EN)
 collate  English_United States.utf8
 ctype    English_United States.utf8
 tz       Europe/Berlin
 date     2026-06-12
 pandoc   3.8.3 @ c:\\Program Files\\Positron\\resources\\app\\quarto\\bin\\tools/ (via rmarkdown)
 quarto   1.9.36 @ C:\\PROGRA~1\\Quarto\\bin\\quarto.exe

─ Packages ───────────────────────────────────────────────────────────────────
 package      * version    date (UTC) lib source
 abind          1.4-8      2024-09-12 [1] CRAN (R 4.5.0)
 backports      1.5.0      2024-05-23 [1] CRAN (R 4.5.0)
 base64enc      0.1-6      2026-02-02 [1] CRAN (R 4.5.2)
 broom          1.0.11     2025-12-04 [1] CRAN (R 4.5.2)
 car            3.1-3      2024-09-27 [1] CRAN (R 4.5.1)
 carData        3.0-5      2022-01-06 [1] CRAN (R 4.5.1)
 cli            3.6

# Execution Time

In [ ]:
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))


R code execution time: 77.98 seconds.

```` markdown
---
title: "Data Preparation"
subtitle: |
  Data Preparation for Analyses
date: last-modified
date-format: MMMM D, YYYY
format:
  html:
    toc: true
    code-fold: true
    code-tools: true
execute:
  echo: true
  warning: true
  eval: true
  message: true
---

Goal: create long format analysis tibble with outcome, predictors, respondent and country IDs.

# Setup {#sec-setup}

quarto-executable-code-5450563D

```r
#| label: setup
#| include: false
# execution time
start_time <- Sys.time()
# console width
options(width = 80)
# packages
p_required <- c(
  "tidyverse", # dplyr, ggplot2, tidyr etc
  "here", # to never worry about directories
  "ggpubr", # theme_pubr()
  "janitor", # tabyl()
  "knitr", # for kable()
  "sessioninfo" # session docs
)
packages <- rownames(installed.packages())
p_to_install <- p_required[!(p_required %in% packages)]
if (length(p_to_install) > 0) {
  install.packages(p_to_install)
}
sapply(p_required, require, character.only = TRUE)
rm(p_required, p_to_install, packages)
```

# Load Data

Full dataset (and pre-processing pipeline) is available at [GitHub](https://github.com/LS-Konig/eu25games2019). We can directly fetch the data from there (by now also compressed, so only ca 30 MB). `read_rds()` can decompress the file directly.

quarto-executable-code-5450563D

```r
#| label: load-data
# set to true do fetch from source repo
download_fresh <- F

if (download_fresh) {
  # URL of raw data
  raw_url <- "https://raw.githubusercontent.com/LS-Konig/eu25games2019/main/data/03_final/eu25games2019.rds?cache_bust=1"

  # Create temporary file path
  tmp <- tempfile(fileext = ".rds")

  # Download binary file locally
  download.file(raw_url, tmp, mode = "wb")

  # Read data into R (takes a couple seconds as it decompresses the file)
  eu25games2019 <- read_rds(tmp)

  # Write (compressed) file to data/01_raw
  write_rds(
    eu25games2019,
    file = here(
      "data",
      "01_raw",
      "eu25games2019.rds"
    ),
    compress = "xz"
  )

  # Clean up variables
  rm(raw_url, tmp)

  write
} else {
  eu25games2019 <- read_rds(
    file = here(
      "data",
      "01_raw",
      "eu25games2019.rds"
    )
  )
}
```

The conjoint partner nationality is now split into the value *shown* on screen
(`cj_<slot>_nationality_shown`) and the background lottery that was *drawn*
(`cj_<slot>_nationality_drawn`, plus its raw string `..._drawn_raw`).
These replace the old conflated `cj_<slot>_nat` / `cj_<slot>_nat_raw`.
The check below confirms the rename landed across all six profile slots and that
no game columns leak into Wave 1.

quarto-executable-code-5450563D

```r
#| label: verify-raw-rename
slots <- c(
  "cj_dict1",
  "cj_dict2",
  "cj_dict3",
  "cj_trust1",
  "cj_trust2",
  "cj_trust3"
)
new_cols <- as.vector(outer(
  slots,
  c("_nationality_drawn", "_nationality_drawn_raw", "_nationality_shown"),
  paste0
))
old_cols <- c(
  paste0(slots, "_nat"),
  paste0(slots, "_nat_raw"),
  paste0("der_", slots, "_nationality")
)

stopifnot(
  # dimensions roughly stable (rename is net-neutral on column count)
  abs(nrow(eu25games2019) - 103685) < 50,
  abs(ncol(eu25games2019) - 847) < 5,
  # 18 new shown/drawn columns present
  all(new_cols %in% names(eu25games2019)),
  # no old conflated names survive
  !any(old_cols %in% names(eu25games2019)),
  # no dictator/trust games in wave 1: all game-slot columns NA there
  # (the separate cj_task* / cj_rand* conjoint instrument does run in wave 1)
  all(vapply(
    eu25games2019[
      eu25games2019$meta_wave == 1,
      grepl("^cj_(dict|trust)[123]_", names(eu25games2019))
    ],
    \(x) all(is.na(x)),
    logical(1)
  ))
)
```


# Filter out wave 1 only respondets

Data comes in three waves with partial panel structure.

quarto-executable-code-5450563D

```r
#| label: explore-partial-panel
# count obs per wave
cat("Raw observations per wave:\n")
eu25games2019 |>
  count(meta_wave)

# total obs (raw)
cat("\nRaw observations total:\n")
eu25games2019 |>
  nrow()

# missing IDs --- would distort every duplicate count below, so check first ----
cat("\nMissing values in ID variables:\n")
eu25games2019 |>
  summarise(
    na_responseid = sum(is.na(meta_responseid)),
    na_psid = sum(is.na(meta_psid)),
    na_pid = sum(is.na(meta_pid))
  )

# distinct ID values -----------------------------------------------------------
cat("\nDistinct `meta_responseid` values:\n")
eu25games2019 |>
  distinct(meta_responseid) |>
  nrow()

cat("\nDistinct `meta_psid` values:\n")
eu25games2019 |>
  distinct(meta_psid) |>
  nrow()

cat("\nDistinct `meta_pid` values:\n")
eu25games2019 |>
  distinct(meta_pid) |>
  nrow()

# row-key integrity: meta_responseid (expected: none) --------------------------
# anything > 0 here is a data-integrity issue, not respondent behaviour
cat("\n`meta_responseid` duplication (expected: none):\n")
eu25games2019 |>
  count(meta_responseid) |>
  filter(n > 1) |>
  summarise(duplicated_ids = n(), excess_rows = sum(n - 1))

# meta_psid duplication --------------------------------------------------------
# psid = sample-instance ID; collisions are usually partial+complete / re-entries
cat("\n`meta_psid` duplication (overall):\n")
eu25games2019 |>
  count(meta_psid) |>
  filter(n > 1) |>
  summarise(duplicated_psids = n(), excess_rows = sum(n - 1))

cat("\n`meta_psid` duplicate rows per wave:\n")
eu25games2019 |>
  count(meta_wave, meta_psid) |>
  filter(n > 1) |>
  summarise(
    duplicated_psids = n(),
    excess_rows = sum(n - 1),
    .by = meta_wave
  ) |>
  arrange(meta_wave)

# does any psid leak across waves? (it shouldn't, if it's per-assignment)
cat("\n`meta_psid` values spanning more than one wave:\n")
eu25games2019 |>
  distinct(meta_psid, meta_wave) |>
  count(meta_psid) |>
  filter(n > 1) |>
  nrow()

# meta_pid WITHIN-wave duplication --- the suspect case ------------------------
# same panelist appearing >1x in a single wave (re-entry / partial+complete)
cat("\n`meta_pid` within-wave duplication, per wave:\n")
eu25games2019 |>
  count(meta_wave, meta_pid) |>
  filter(n > 1) |>
  summarise(duplicated_pids = n(), excess_rows = sum(n - 1), .by = meta_wave) |>
  arrange(meta_wave)

cat("\n`meta_pid` within-wave duplication (total across waves):\n")
eu25games2019 |>
  count(meta_wave, meta_pid) |>
  filter(n > 1) |>
  summarise(duplicated_pids = n(), excess_rows = sum(n - 1))

# cross-wave participation --- EXPECTED panel overlap, NOT duplication ---------
cat("\nPanelists by number of distinct waves participated:\n")
eu25games2019 |>
  distinct(meta_pid, meta_wave) |>
  count(meta_pid, name = "n_waves") |>
  count(n_waves, name = "n_panelists") |>
  arrange(n_waves)

cat("\nPanelists appearing in 2+ waves:\n")
eu25games2019 |>
  distinct(meta_pid, meta_wave) |>
  count(meta_pid, name = "n_waves") |>
  filter(n_waves > 1) |>
  nrow()
```

# Filter our respondents who did not participate in games (w2 and w3 only)

quarto-executable-code-5450563D

```r
#| label: filter-out-nongame-respondents
# pids of anyone who participated in at least one of w2 / w3
pids_in_games <- eu25games2019 |>
  filter(meta_wave %in% c(2, 3)) |>
  distinct(meta_pid) |>
  pull(meta_pid)

# keep all w2 + w3 rows; from w1 keep only pids that reappear in w2 or w3
eu25games2019 <- eu25games2019 |>
  # n = 103,685 -> 70,685 (dropped 33,000: wave-1-only respondents who never played a game)
  filter(meta_wave %in% c(2, 3) | meta_pid %in% pids_in_games)

cat("Observations remaining:\n")
nrow(eu25games2019)

cat("\nObservations per wave:\n")
eu25games2019 |>
  count(meta_wave)

cat("\nDistinct panelists remaining:\n")
eu25games2019 |>
  distinct(meta_pid) |>
  nrow()
```

# Select clean covariate candidates

quarto-executable-code-5450563D

```r
#| label: inspect-cleaned-vars
eu25games2019 |>
  select(-contains("_raw")) |>
  select(contains("q_")) |>
  glimpse()
```

**NOTE:** We have decided to only include items asked *prior* to the games.

We can keep all questions asked in wave 1, as those games only started in wave 2.

quarto-executable-code-5450563D

```r
#| label: inspect-w1-vars
eu25games2019 |>
  filter(meta_wave == 1) |>
  select(-contains("_raw")) |>
  select(contains("q_")) |>
  glimpse()
```

These can serve as fillup maybe.

Variables in w2 and w3 before games (Source: [Data GitHub Repo](https://github.com/LS-Konig/eu25games2019)):

quarto-executable-code-5450563D

```r
#| label: vars-pre-game
vars_pre_game <- c(
  "q_gender",
  "q_age",
  "q_nationality",
  "q_identity_country",
  "q_identity_eu",
  "q_identity_europe",
  "q_religion",
  "q_class",
  "q_euint_pos",
  "q_eu_efficacy_understand",
  "q_eu_efficacy_care",
  "q_eu_efficacy_nosay",
  "q_pop_will",
  "q_pop_people",
  "q_pop_citizen",
  "q_pop_reps",
  "q_pop_goodevil",
  "q_pop_compromise",
  "q_dem_compromise",
  "q_dem_listen",
  "q_tech_experts",
  "q_tech_leaders",
  "q_party_harm",
  "q_people_incompetent",
  "q_eu_longterm",
  "q_eu_responsive",
  "q_eu_imp_nat_econ",
  "q_eu_imp_nat_cul",
  "q_eu_imp_nat_pol",
  "q_eu_abolish",
  "q_eu_satisfaction",
  "q_election_challenge_polgroups",
  "q_election_impact_eu",
  "q_election_importance_self"
)

# ignoring vote choice and party IDs for now
```

| Variable | Coverage | Question wording |
|:---|:---:|:---|
| `q_gender` | 2/3 | Please indicate your gender. |
| `q_age` | 2/3 | How old are you? |
| `q_nationality` | 2/3 | What is your nationality? Please tell me the country(ies) that applies(y). |
| `q_identity_country` | 2/3 | Please tell me how attached you feel to... - United Kingdom |
| `q_identity_eu` | 2/3 | Please tell me how attached you feel to... - The European Union (EU) |
| `q_identity_europe` | 2/3 | Please tell me how attached you feel to... - Europe |
| `q_religion` | 2/3 | Do you consider yourself to be…? |
| `q_class` | 2/3 | Do you see yourself and your household belonging to...? |
| `q_euint_pos` | 2/3 | Some people say that European integration should be pushed further. Others say it already has gone too far. What is your opinion? Please indicate your views using a scale from 1 to 7, where '1' means European integration "has already gone too far" and '7' means it "should be pushed further". What number on this scale best describes your position? |
| `q_eu_efficacy_understand` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - I can understand and evaluate important political questions about the EU |
| `q_eu_efficacy_care` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - The politicians in the EU care about what people like me think |
| `q_eu_efficacy_nosay` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - People like me don't have any say about what the EU does |
| `q_pop_will` | 2/3 | Please indicate how much you agree with each of the following statements. - The politicians should follow the will of the people. |
| `q_pop_people` | 2/3 | Please indicate how much you agree with each of the following statements. - The people, and not politicians, should make our most important policy decisions. |
| `q_pop_citizen` | 2/3 | Please indicate how much you agree with each of the following statements. - I would rather be represented by a citizen than by a specialized politician. |
| `q_pop_reps` | 2/3 | Please indicate how much you agree with each of the following statements. - Elected officials talk too much and take too little action. |
| `q_pop_goodevil` | 2/3 | Please indicate how much you agree with each of the following statements. - Politics is ultimately a struggle between good and evil. |
| `q_pop_compromise` | 2/3 | Please indicate how much you agree with each of the following statements. - What people call "compromise" in politics is really just selling out on one's principles. |
| `q_dem_compromise` | 2/3 | Please indicate how much you agree with each of the following statements. - In a democracy it is important to make compromises among differing viewpoints. |
| `q_dem_listen` | 2/3 | Please indicate how much you agree with each of the following statements. - It is important to listen to the opinion of other groups. |
| `q_tech_experts` | 2/3 | Please indicate how much you agree with each of the following statements. - Our country would run better if political decisions were left up to experts instead of politicians and citizens. |
| `q_tech_leaders` | 2/3 | Please indicate how much you agree with each of the following statements. - The leaders of our country should be more educated and skilled than ordinary citizens. |
| `q_party_harm` | 2/3 | Please indicate how much you agree with each of the following statements. - Political parties do more harm than good to society |
| `q_people_incompetent` | 2/3 | Please indicate how much you agree with each of the following statements. - Ordinary people don't know what policies are good for them. |
| `q_eu_longterm` | 2/3 | In thinking about the European Union (EU) and its decisions more generally, some people argue that it should act upon its long-term goals without pandering to the demand of the people. On the other hand, others argue that the EU should listen to the demand of people, rather than to act upon its long-term goals. When it comes to the question of whether the EU should act upon its long-term goals or it should listen to the demand of the people, what is your opinion? - The EU should act upon its long-term goals |
| `q_eu_responsive` | 2/3 | In thinking about the European Union (EU) and its decisions more generally, some people argue that it should act upon its long-term goals without pandering to the demand of the people. On the other hand, others argue that the EU should listen to the demand of people, rather than to act upon its long-term goals. When it comes to the question of whether the EU should act upon its long-term goals or it should listen to the demand of the people, what is your opinion? - The EU should listen to the demand of the people |
| `q_eu_imp_nat_econ` | 2/3 | Could you tell me whether you think that the EU has a rather positive or rather negative effect on each of the following domains? - UK economy |
| `q_eu_imp_nat_cul` | 2/3 | Could you tell me whether you think that the EU has a rather positive or rather negative effect on each of the following domains? - Our culture and identity |
| `q_eu_imp_nat_pol` | 2/3 | Could you tell me whether you think that the EU has a rather positive or rather negative effect on each of the following domains? - UK's political status and influence |
| `q_eu_abolish` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - If the EU started making a lot of decisions that most people disagreed with, it might be better to do away with the EU altogether |
| `q_eu_satisfaction` | 2/3 | Please tell me to what extent you agree or disagree with each of the following statements. - I am satisfied with the performance of the EU |
| `q_election_challenge_polgroups` | 2 | What is your opinion? - How likely do you think the upcoming elections will challenge the existing political groups and members of the European Parliament? |
| `q_election_impact_eu` | 2 | What is your opinion? - How likely do you think the upcoming elections will play a big role in shaping the European Parliament and the EU's next five years? |
| `q_election_importance_self` | 2 | How important is the outcome of the upcoming election to you personally? |

: Pre-game (wave 2/3) respondent-level survey items considered as covariate candidates. Coverage indicates the waves in which the item was asked. {#tbl-pre-game-vars}

# Clean incomplete responses, failed attention checks

quarto-executable-code-5450563D

```r
#| label: survey-cleaning
eu25games2019_clean <- eu25games2019 |>
  # only respondents who passed attention check
  # n = 70,685 -> 41,457 (dropped 29,228: failed attention check der_att_check_3)
  filter(der_att_check_3)

# still pid duplicates?
eu25games2019_clean |>
  group_by(meta_wave) |>
  count(meta_pid) |>
  count(n)

# yup

# and with completed games?
eu25games2019_clean |>
  filter(!is.na(cj_dict1_pl2)) |>
  group_by(meta_wave) |>
  count(meta_pid) |>
  count(n)

# still a few

eu25games2019_clean |>
  filter(!is.na(cj_dict1_pl2)) |>
  group_by(meta_wave) |>
  count(meta_pid, sort = T) |>
  filter(n > 1)

# one estonian PID contains 8 (10 w/ games) rows, demographics point to different people, might be true and due to bug of invitation link (identical PSIDs as well), but cannot be certain, hence remove

# others include mostly 2, so I suspect people stopped the survey and joined later again.
# use completeness to filter

eu25games2019_clean |>
  filter(!is.na(cj_dict1_pl2)) |>
  group_by(meta_wave, meta_pid) |>
  filter(meta_finished) |>
  count(meta_pid, sort = T) |>
  filter(n > 1)

# this solves most, but 8 PIDs (9 minus the Estonian quirk) remain

# id 1275872150 is same person according to demographics but gave different responses
# is panellist across all 3 waves but double wave 2 responses (Dutch),
# starting time differs only by seconds which is sus
# which response has more missings?
eu25games2019_clean |>
  filter(meta_wave == 2, meta_pid == "1275872150") |>
  group_by(meta_responseid) |>
  summarise(total_na = sum(is.na(across(everything()))))
# almost identical

# id 1385201604 also strange, one response started earlier and ended later than the other,
# responses themselves very similar again
eu25games2019_clean |>
  filter(meta_wave == 2, meta_pid == "1385201604") |>
  group_by(meta_responseid) |>
  summarise(total_na = sum(is.na(across(everything()))))

# suspection: they might have clicked the invite link twice, and it worked (even though it shouldn't have)
eu25games2019_clean |>
  filter(meta_wave == 2, meta_pid == "1416296382") |>
  group_by(meta_responseid) |>
  summarise(total_na = sum(is.na(across(everything()))))
# same picture, responses very very similar

# response times just are odd, as if the same person did it twice

# choice: remove them to be safe

# collect IDs to remove
to_remove <- eu25games2019_clean |>
  filter(!is.na(cj_dict1_pl2)) |>
  group_by(meta_wave, meta_pid) |>
  filter(meta_finished) |>
  count(meta_pid, sort = T) |>
  filter(n > 1) |>
  pull(meta_pid)

# implement "cleaning" steps
eu25games2019_clean2 <- eu25games2019_clean |>
  # only complete responses
  # n = 41,457 -> 36,912 (dropped 4,545: incomplete responses)
  filter(meta_finished) |>
  # remove identified specialists
  # n = 36,912 -> 36,881 (dropped 31: duplicate-PID specialists with completed games)
  filter(!(meta_pid %in% to_remove)) |>
  # keep only respondents who appear in w2 or w3, including their w1 rows if they exist
  # n = 36,881 -> 35,235 (dropped 1,646: pids never present in w2/w3)
  filter(meta_pid %in% meta_pid[meta_wave %in% c(2, 3)]) |>
  # remove duplicate IDs in w1
  # n = 35,235 -> 35,212 (dropped 23: three duplicate w1 IDs)
  filter(!(meta_wave == 1 & meta_pid == "1078674746")) |>
  filter(!(meta_wave == 1 & meta_pid == "1277956255")) |>
  filter(!(meta_wave == 1 & meta_pid == "1516614375")) |>
  # remove NA IDs in w1
  # n = 35,212 -> 35,212 (dropped 0: no NA w1 IDs remain after prior filters)
  filter(!(meta_wave == 1 & is.na(meta_pid)))
```

# Inspect missings in selected covariates

Because the games only start in wave 2, every pre-game item asked in wave 1 is a valid
pre-treatment measure for the same panelist.
We therefore carry each panelist's wave-1 value forward into their *missing* wave-2/3 cells,
matched on `(meta_pid, meta_country)`.
Only `NA` cells are touched — existing values are never overwritten — and every filled cell is
flagged so the operation is fully auditable.
The recovery is almost entirely attitudinal items (EU-efficacy and populism scales): carrying
them forward assumes temporal stability, which is acceptable here because the waves are close
together and these enter the analysis only as respondent-level moderators.

quarto-executable-code-5450563D

```r
#| label: missing-values
# w1 items available as fill source
eu25games2019_clean2 |>
  filter(meta_wave == 1) |>
  select(all_of(vars_pre_game)) |>
  glimpse()

# analysis selection: ids + pre-game covars + conjoint vars
eu25games2019_clean_selected <- eu25games2019_clean2 |>
  select(
    meta_wave,
    meta_country,
    meta_pid,
    all_of(vars_pre_game),
    starts_with("cj_attr"),
    starts_with("cj_treatment"),
    starts_with("cj_dict"),
    starts_with("cj_trust"),
    # postgame demographic vars only
    q_rural_urban,
    q_edu_age_stop
  )

# step 1: each panelist's w1 covariate values
w1_values <- eu25games2019_clean_selected |>
  filter(meta_wave == 1) |>
  select(meta_pid, meta_country, all_of(vars_pre_game))

# step 2: missing counts before backfill
na_before <- eu25games2019_clean_selected |>
  filter(meta_wave %in% c(2, 3)) |>
  summarise(across(all_of(vars_pre_game), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "na_before")

# step 3: backfill one variable from its w1 partner
backfill_w1 <- function(df, v) {
  w1 <- paste0(v, "_w1")
  df |>
    # flag fillable cell
    mutate(
      "filled_{v}" := meta_wave %in%
        c(2, 3) &
        is.na(.data[[v]]) &
        !is.na(.data[[w1]])
    ) |>
    # carry w1 value into NA cell
    mutate("{v}" := coalesce(.data[[v]], .data[[w1]]))
}

# attach w1 values, fold the backfill over every covariate, drop helpers
eu25games2019_clean_selected <- eu25games2019_clean_selected |>
  left_join(
    w1_values,
    by = c("meta_pid", "meta_country"),
    suffix = c("", "_w1")
  ) |>
  reduce(vars_pre_game, backfill_w1, .init = _) |>
  select(-ends_with("_w1"))

# step 4: missing counts after backfill
na_after <- eu25games2019_clean_selected |>
  filter(meta_wave %in% c(2, 3)) |>
  summarise(across(all_of(vars_pre_game), ~ sum(is.na(.x)))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "na_after")

# step 5: cells filled per variable
n_filled <- eu25games2019_clean_selected |>
  filter(meta_wave %in% c(2, 3)) |>
  summarise(across(starts_with("filled_"), ~ sum(.x))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "n_filled") |>
  mutate(variable = str_remove(variable, "^filled_"))

# step 6: audit table
fill_audit <- na_before |>
  left_join(na_after, by = "variable") |>
  left_join(n_filled, by = "variable") |>
  filter(n_filled > 0) |>
  arrange(desc(n_filled))

fill_audit |>
  kable(
    caption = "Wave-1 backfill: W2/W3 missing cells before/after and cells recovered."
  )

cat("Total cells filled from wave 1:", sum(fill_audit$n_filled), "\n")

# step 7: drop flag columns from analysis object
eu25games2019_clean_selected <- eu25games2019_clean_selected |>
  select(-starts_with("filled_"))

# verification
stopifnot(
  # one row per panelist per wave
  nrow(filter(
    count(
      filter(eu25games2019_clean_selected, !is.na(meta_pid)),
      meta_wave,
      meta_pid
    ),
    n > 1
  )) ==
    0,
  # accounting reconciles
  all(fill_audit$na_after == fill_audit$na_before - fill_audit$n_filled)
)

# inspect remaining NAs
eu25games2019_clean_selected |>
  summary()

# problematic covariates to be dropped because of many NAs:
# q_euint_pos (32% NA)
# q_eu_efficacy_care (41% NA)
# q_eu_efficacy_nosay (41% NA)
# q_pop_will (26% NA)
# q_pop_people (26% NA)
# q_pop_citizen (26% NA)
# q_election_challenge_polgroups (54% NA)
# q_election_impact_eu (54% NA)
# q_election_importance_self (54% NA)

eu25games2019_clean_selected_final <- eu25games2019_clean_selected |>
  select(
    -q_euint_pos,
    -q_eu_efficacy_care,
    -q_eu_efficacy_nosay,
    -q_pop_will,
    -q_pop_people,
    -q_pop_citizen,
    -q_election_challenge_polgroups,
    -q_election_impact_eu,
    -q_election_importance_self,
    -q_nationality # correlates with meta_country
  ) |>
  # n = 35,212 -> 28,201 (dropped 7,011: NA cleaning -- rows with any remaining missing covariate)
  na.omit()
```

# Reshape to long format data for analyses

`eu25games2019` contains data in wide format, i.e., one row corresponds one survey response. To analyze outcomes, the data need to be transformed to long (tidy) format, with one row equal to one observation, in this case one row should be one game played.

quarto-executable-code-5450563D

```r
#| label: game-outcomes-to-long
# select relevant variables
eu25_long <- eu25games2019_clean_selected_final |>
  # to long format
  # n = 28,201 -> 169,206 (reshape, not a drop: x6 = 2 games x 3 rounds; 84,603 rows per game)
  pivot_longer(
    cols = -c(
      starts_with("meta_"),
      starts_with("q_"),
      starts_with("cj_attr"),
      starts_with("cj_treatment"),
      contains("_instr_")
    ),
    names_to = c("game_type", "round", ".value"),
    names_pattern = "^(cj_dict|cj_trust)(\\d+)_(.+)$"
  ) |>
  # keep cj_ prefix for experimental vars
  rename_with(
    ~ paste0("cj_", .),
    -c(
      starts_with("meta_"),
      starts_with("q_"),
      starts_with("cj_attr"),
      starts_with("cj_treatment"),
      contains("_instr_")
    )
  )
```

# Display-aware profile covariates

The four treatment conditions in `cj_treatment` govern *which* profile features were
actually shown to the respondent.
EU stance was displayed only in conditions whose name contains `eustance` (treatments 1
and 3); partisan affiliation was displayed only in conditions whose name contains
`partisan` (treatments 1 and 2).
The raw conjoint columns `cj_eupos` and `cj_party_raw` carry a behind-the-scenes value
in *every* condition, including those where the feature was never displayed, so the
displayed contrast has to be reconstructed from `cj_treatment`.
The partner nationality is the displayed `cj_nationality_shown` (not the background
`cj_nationality_drawn` lottery, which is overridden by the treatment branch in every
condition except `4_somenat`).

We therefore derive two display-aware profile covariates with the `der_cj_` prefix.
`der_cj_eu_identity` collapses to three levels: the shown EU-citizen contrast where EU
stance was displayed, and a single `not_displayed` level otherwise.
`der_cj_partisanship` contrasts conational profiles where party was shown (`shown`,
treatments 1 and 2) against the apples-to-apples conational profile where party was
suppressed (`not_shown`, treatment 4 with the displayed nationality equal to the
respondent's country, i.e. `cj_nationality_shown == "own_country"`); every other profile
is `not_applicable`.

quarto-executable-code-5450563D

```r
#| label: display-aware-covariates
eu25_long <- eu25_long |>
  mutate(
    # eu stance shown -> 1,3
    der_cj_eu_identity = case_when(
      grepl("eustance", cj_treatment) & cj_eupos == "eu_citizen" ~ "eu_citizen",
      grepl("eustance", cj_treatment) & cj_eupos == "not_eu_citizen" ~
        "not_eu_citizen",
      !grepl("eustance", cj_treatment) ~ "not_displayed"
    ),
    # party shown -> 1,2; conational suppressed -> 4 & own_country
    der_cj_partisanship = case_when(
      grepl("partisan", cj_treatment) ~ "shown",
      cj_treatment == "4_somenat" &
        cj_nationality_shown == "own_country" ~ "not_shown",
      .default = "not_applicable"
    )
  )
```

The cross-tabs below confirm the display logic: `der_cj_eu_identity` is `not_displayed`
in exactly the two no-`eustance` conditions, and `der_cj_partisanship` separates the
shown-party conationals from the treatment-4 suppressed conationals.

quarto-executable-code-5450563D

```r
#| label: tbl-display-aware-check
count(eu25_long, cj_treatment, der_cj_eu_identity) |>
  kable(caption = "der_cj_eu_identity by treatment condition.")

count(eu25_long, cj_treatment, cj_nationality_shown, der_cj_partisanship) |>
  kable(
    caption = "der_cj_partisanship by treatment condition and shown nationality."
  )
```

quarto-executable-code-5450563D

```r
#| label: verify-display-aware
stopifnot(
  # no missingness introduced
  !anyNA(eu25_long$der_cj_eu_identity),
  !anyNA(eu25_long$der_cj_partisanship),
  # not_displayed iff eu stance not shown
  all(
    (eu25_long$der_cj_eu_identity == "not_displayed") ==
      !grepl("eustance", eu25_long$cj_treatment)
  ),
  # shown iff party displayed condition
  all(
    (eu25_long$der_cj_partisanship == "shown") ==
      grepl("partisan", eu25_long$cj_treatment)
  ),
  # not_shown is exactly the suppressed conational counterfactual
  all(
    (eu25_long$der_cj_partisanship == "not_shown") ==
      (eu25_long$cj_treatment == "4_somenat" &
        eu25_long$cj_nationality_shown == "own_country")
  )
)
```

The shown nationality is treatment-determined: co-national in the two conational
conditions, EU in the EU-stance condition, and equal to the drawn lottery only in
`4_somenat`. This is the displayed-profile contrast the moderator analyses use, in
contrast to the flat drawn lottery.

quarto-executable-code-5450563D

```r
#| label: tbl-shown-vs-drawn-check
count(eu25_long, cj_treatment, cj_nationality_shown) |>
  kable(
    caption = "Shown nationality by treatment condition (now treatment-dependent)."
  )
```

quarto-executable-code-5450563D

```r
#| label: verify-shown-vs-drawn
shown <- eu25_long$cj_nationality_shown
drawn <- eu25_long$cj_nationality_drawn
trt <- eu25_long$cj_treatment

stopifnot(
  # conational conditions display own_country
  all(shown[grepl("conat", trt)] == "own_country"),
  # eu-stance-only condition displays eu
  all(shown[trt == "3_eunat_eustance"] == "eu"),
  # somenat passes the drawn lottery through unchanged
  all(shown[trt == "4_somenat"] == drawn[trt == "4_somenat"]),
  # shown is no longer the flat lottery: it varies by treatment
  dplyr::n_distinct(shown[grepl("conat", trt)]) == 1
)
```

# Descriptives

quarto-executable-code-5450563D

```r
#| label: tbl-descriptives
# respondent covariates --------------------------------------------------------
# gender
skimr::skim(eu25_long)
```

# Save long format data

quarto-executable-code-5450563D

```r
#| label: save-data
write_rds(
  eu25_long,
  file = here(
    "data",
    "02_processed",
    "eu25_long.rds"
  ),
  compress = "xz"
)
```

# Session Info {#sec-session-info .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: session-info
#| eval: true
#| echo: true
session_info()
```

# Execution Time {#sec-exec-time .appendix .unnumbered}

quarto-executable-code-5450563D

```r
#| label: exec-time
#| eval: true
#| echo: true
#| include: true
end_time <- Sys.time()
exec_time <- end_time - start_time
cat(paste(
  "R code execution time:",
  round(as.numeric(exec_time, units = "secs"), 2),
  "seconds."
))
```
````